In [17]:
import os 
from dotenv import load_dotenv

import pandas as pd

from langchain.chat_models import init_chat_model

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.csv_loader import CSVLoader

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_chroma import Chroma
from langchain_core.tools import tool





In [18]:
load_dotenv(override=True)

True

In [19]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

In [30]:
model = init_chat_model(
    "groq:qwen/qwen3.6-27b",
    # Kwargs passed to the model:
)


In [31]:
response = model.invoke("hi")

In [32]:
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user said "hi". This is a simple greeting.\n2.  **Identify Intent:** The user is initiating a conversation. No specific question or task is provided.\n3.  **Determine Response Strategy:** \n   - Acknowledge the greeting warmly\n   - Keep it open-ended to encourage further interaction\n   - Offer assistance\n   - Maintain a friendly, professional tone\n4.  **Draft Response (Mental):** "Hello! How can I help you today?" or "Hi there! What\'s on your mind? I\'m here to help with anything you need."\n5.  **Refine Response:** Keep it concise and welcoming. "Hi! How can I assist you today?" works well. It\'s polite, open, and invites the user to share their needs.\n6.  **Final Output Generation:** Match the refined response.✅\n</think>\n\nHi! How can I help you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 210, 'prompt_tokens': 11, 'total_tokens': 221, 'comp

## Data ingestion

### Data loader

In [33]:
class AnimeDataLoader:
    def __init__(self,original_csv:str,processed_csv:str): # both are paths 
        self.original_csv = original_csv
        self.processed_csv=processed_csv

    def load_process(self):

        df = pd.read_csv(

            self.original_csv,
            encoding="utf-8",
            on_bad_lines="skip"
        ).dropna()


        required_col = {"Name","Genres","sypnopsis"}
        if not required_col.issubset(df.columns):
            raise ValueError("Required columns are missing")


        df['combined_info'] = (
            "Title: " + df["Name"] + "\n" +
            "Overview: " + df["sypnopsis"] + "\n" +
            "Genres: " + df["Genres"]
        )

        df[["combined_info"]].to_csv(
            self.processed_csv,
            index=False,
            encoding="utf-8"
        )

        return self.processed_csv



In [34]:
import chromadb


class VectorStore:
    def __init__(self,csv_location:str,persitent_dir:str):
        self.csv_location=csv_location
        self.persitent_dir=persitent_dir

        self.embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


    def build_and_save_vector_store(self,):
         loader = CSVLoader(
             file_path=self.csv_location,
             encoding="utf-8",
             metadata_columns=[]
         )

         documents = loader.load()

         # Store whole anime entries without splitting rows
         db = Chroma.from_documents(
             documents,
             self.embedding,
             persist_directory=self.persitent_dir
         )


    def load_vector_store(self):
        return Chroma(
            persist_directory=self.persitent_dir
            ,embedding_function=self.embedding
        )


In [35]:
print("Buiding vector db")

original_csv = "anime_with_synopsis.csv"
processed_csv = "anime_updated.csv"
persistent_dir = "chroma_db"



processed_data = AnimeDataLoader(original_csv,processed_csv)
processed_csv_path = processed_data.load_process()
vecotr_builder = VectorStore(processed_csv_path,persistent_dir)

vecotr_builder.build_and_save_vector_store()
print("vector database ready")




Buiding vector db


vector database ready


In [36]:
vecotr_builder = VectorStore(processed_csv_path,persistent_dir)

retriever = vecotr_builder.load_vector_store().as_retriever(search_kwargs={"k": 10})

In [37]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# -----------------------------------
# Load Vector Store
# -----------------------------------
vector_builder = VectorStore(processed_csv_path, persistent_dir)

retriever = vector_builder.load_vector_store().as_retriever(search_kwargs={"k": 10})


# -----------------------------------
# Tool
# -----------------------------------
@tool
def anime_retriever_tool(query: str) -> str:
    """
    Search the anime vector database.
    """

    documents = retriever.invoke(query)

    if not documents:
        return "No relevant anime found."

    return "\n\n".join(doc.page_content for doc in documents)


# -----------------------------------
# Bind tool
# -----------------------------------
model_with_tools = model.bind_tools([anime_retriever_tool])


# -----------------------------------
# User Message
# -----------------------------------
messages = [
    HumanMessage(
        content="Can you suggest me best action anime?"
    )
]


# -----------------------------------
# Step 1 : Model decides whether to use tool
# -----------------------------------
ai_msg = model_with_tools.invoke(messages)

messages.append(ai_msg)


# -----------------------------------
# Step 2 : Execute tool(s)
# -----------------------------------
for tool_call in ai_msg.tool_calls:

    tool_result = anime_retriever_tool.invoke(tool_call["args"])

    messages.append(
        ToolMessage(
            content=tool_result,
            tool_call_id=tool_call["id"]
        )
    )


# -----------------------------------
# Step 3 : Final answer
# -----------------------------------
if ai_msg.tool_calls:
    final_response = model_with_tools.invoke(messages)
    print(final_response.content)
else:
    print(ai_msg.content)

In [38]:
print("Tool Calls:", ai_msg.tool_calls)
print("Tool Result:", tool_result if "tool_result" in locals() else "No tool executed")
if "final_response" in locals():
    print("Final Response:", final_response)


Tool Calls: [{'name': 'anime_retriever_tool', 'args': {'query': 'best action anime'}, 'id': 'qfyts50m9', 'type': 'tool_call'}]
Tool Result: Action, Comedy, Historical, Samurai, Shounen

Action, Comedy, Historical, Samurai, Shounen

with no clear sides... (Source: Anime-Planet)Genres: Action, Mecha, Sci-Fi, Shounen, Space

with no clear sides... (Source: Anime-Planet)Genres: Action, Mecha, Sci-Fi, Shounen, Space

Action, Comedy, Ecchi, Harem, Super Power

Action, Comedy, Ecchi, Harem, Super Power

Action, Adventure, Super Power, Shounen

Action, Adventure, Super Power, Shounen

Action, Historical, Drama, Romance, Martial Arts, Samurai, Shounen

Action, Historical, Drama, Romance, Martial Arts, Samurai, Shounen
Final Response: content='' additional_kwargs={'reasoning_content': 'The user is asking for suggestions for "best action anime".\nI have retrieved a list of genres and potentially some descriptions, but the tool output is a bit abstract, just listing genres associated with search r

In [41]:
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(content="Can you suggest me some naruto like  animes?")
]

# Step 1: Tool decision
ai_msg = model_with_tools.invoke(messages)

if ai_msg.tool_calls:

    tool_call = ai_msg.tool_calls[0]

    tool_result = anime_retriever_tool.invoke(tool_call["args"])

    # Step 2: Give tool output to normal model
    final_prompt = f"""
Answer the user's question using ONLY the following retrieved information.

User Question:
{messages[0].content}

Retrieved Information:
{tool_result}
"""

    final_response = model.invoke(final_prompt)

    print(final_response.content)


<think>
Here's a thinking process:

1.  **Analyze User Question:** The user asks for anime recommendations similar to "Naruto".
2.  **Analyze Retrieved Information:**
   - The retrieved info contains multiple entries, mostly about "Naruto" itself (title, overview, genres: Action, Adventure, Comedy, Super Power, Martial Arts, Shounen).
   - There's also an entry for "Ninin ga Shinobuden" (Overview: school girl meets ninja-in-training, comedy/parody/martial arts genres).
   - There are some fragmented entries about "with no clear sides... Genres: Action, Mecha, Sci-Fi, Shounen, Space" but no title is clearly associated with it in the provided text. It seems like corrupted/fragmented data.
   - The prompt strictly says: "Answer the user's question using ONLY the following retrieved information."
3.  **Identify Constraints & Challenges:**
   - I must ONLY use the provided text.
   - The provided text is heavily focused on Naruto itself and one other anime: "Ninin ga Shinobuden".
   - It l